# 🤖 Predicción de Conversiones en Campañas de Marketing

**Objetivo:** Construir un modelo de clasificación que prediga si una campaña en redes sociales logrará una **tasa de conversión alta** (≥ 4%), permitiendo al equipo de marketing tomar decisiones de inversión más informadas antes de lanzar una campaña.

**Tipo de problema:** Clasificación binaria  
**Modelos evaluados:** Logistic Regression · Random Forest · Gradient Boosting  
**Métricas clave:** Accuracy · Precision · Recall · F1 · ROC-AUC

---

## 1. Importaciones y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
SEED = 42
print('✅ Librerías cargadas')

## 2. Generación del Dataset

In [ ]:
np.random.seed(SEED)
n = 2000

plataformas  = ['Instagram', 'Facebook', 'TikTok', 'LinkedIn']
tipos        = ['Awareness', 'Conversión', 'Retargeting', 'Engagement']
segmentos    = ['18-24', '25-34', '35-44', '45-54']
dispositivos = ['mobile', 'desktop', 'tablet']

plat_col = np.random.choice(plataformas, n, p=[0.35, 0.30, 0.25, 0.10])
tipo_col = np.random.choice(tipos, n)
seg_col  = np.random.choice(segmentos, n)
disp_col = np.random.choice(dispositivos, n, p=[0.65, 0.28, 0.07])

presupuesto = np.random.choice([500, 1000, 2000, 5000, 10000], n, p=[0.30, 0.30, 0.20, 0.15, 0.05])

# CTR influenciado por plataforma
ctr_base = {'Instagram':3.2, 'Facebook':2.5, 'TikTok':4.8, 'LinkedIn':1.8}
ctr = np.array([max(0.2, np.random.normal(ctr_base[p], 0.9)) for p in plat_col])

impresiones  = (presupuesto * np.random.uniform(60, 140, n)).astype(int)
clics        = (impresiones * ctr / 100).astype(int)
duracion_dias = np.random.choice([7, 14, 21, 30], n, p=[0.20, 0.35, 0.25, 0.20])
num_creativos = np.random.choice([1, 2, 3, 4, 5], n, p=[0.10, 0.25, 0.35, 0.20, 0.10])
tiene_video   = np.random.choice([0, 1], n, p=[0.45, 0.55])

# Tasa de conversión real (variable objetivo base)
conv_base = {'Instagram':4.1, 'Facebook':3.5, 'TikTok':2.8, 'LinkedIn':6.2}
tipo_bonus = {'Retargeting':1.5, 'Conversión':1.2, 'Engagement':0.9, 'Awareness':0.7}
seg_bonus  = {'25-34':1.2, '35-44':1.1, '18-24':1.0, '45-54':0.9}

tasa_conv = np.array([
    max(0.1, np.random.normal(
        conv_base[plat_col[i]] * tipo_bonus[tipo_col[i]] * seg_bonus[seg_col[i]]
        + (0.5 if tiene_video[i] else 0)
        + (num_creativos[i] * 0.15),
        1.2
    ))
    for i in range(n)
])

conversiones = (clics * tasa_conv / 100).astype(int)
costo_conv   = np.where(conversiones > 0, presupuesto / conversiones, presupuesto)

# Etiqueta: 1 = tasa de conversión alta (≥ 4%)
target = (tasa_conv >= 4.0).astype(int)

df = pd.DataFrame({
    'plataforma':   plat_col,
    'tipo_campana': tipo_col,
    'segmento_edad':seg_col,
    'dispositivo':  disp_col,
    'presupuesto':  presupuesto,
    'impresiones':  impresiones,
    'clics':        clics,
    'ctr':          ctr.round(2),
    'duracion_dias':duracion_dias,
    'num_creativos':num_creativos,
    'tiene_video':  tiene_video,
    'conversiones': conversiones,
    'alta_conversion': target
})

print(f'Dataset: {df.shape}  |  Balance de clases: {target.mean()*100:.1f}% positivos')
df.head()

## 3. Análisis Exploratorio Previo al Modelado

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Balance de clases
vals = df['alta_conversion'].value_counts()
axes[0].pie(vals, labels=['Conversión Baja', 'Conversión Alta'],
            autopct='%1.1f%%', colors=['#ef4444','#22c55e'], startangle=90)
axes[0].set_title('Balance de Clases')

# Conversión alta por plataforma
conv_plat = df.groupby('plataforma')['alta_conversion'].mean() * 100
axes[1].bar(conv_plat.index, conv_plat.values, color=['#E1306C','#1877F2','#010101','#0A66C2'])
axes[1].set_title('% Conversión Alta por Plataforma')
axes[1].set_ylabel('%')
for i, v in enumerate(conv_plat.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

# CTR vs target
sns.boxplot(data=df, x='alta_conversion', y='ctr',
            palette={0:'#ef4444', 1:'#22c55e'}, ax=axes[2])
axes[2].set_title('Distribución CTR por Clase')
axes[2].set_xticklabels(['Conversión Baja', 'Conversión Alta'])
axes[2].set_ylabel('CTR (%)')

plt.suptitle('Análisis Exploratorio', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Preparación de Features

In [ ]:
# Encoding de variables categóricas
df_model = df.copy()
cat_cols = ['plataforma', 'tipo_campana', 'segmento_edad', 'dispositivo']
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Feature engineering
df_model['clics_por_dia']    = df_model['clics'] / df_model['duracion_dias']
df_model['cpm']              = (df_model['presupuesto'] / df_model['impresiones'] * 1000).round(3)
df_model['creativos_x_dia']  = df_model['num_creativos'] / df_model['duracion_dias']

FEATURES = [c for c in df_model.columns if c not in ['alta_conversion', 'conversiones']]
X = df_model[FEATURES]
y = df_model['alta_conversion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=SEED, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features totales: {len(FEATURES)}')

## 5. Entrenamiento y Evaluación de Modelos

In [ ]:
modelos = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, random_state=SEED))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8,
                                             random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                                    max_depth=4, random_state=SEED)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
resultados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    cv_scores = cross_val_score(modelo, X_train, y_train, cv=cv, scoring='roc_auc')
    resultados[nombre] = {
        'modelo':   modelo,
        'y_pred':   y_pred,
        'y_proba':  y_proba,
        'roc_auc':  roc_auc_score(y_test, y_proba),
        'cv_auc':   cv_scores.mean(),
        'cv_std':   cv_scores.std()
    }
    print(f"[{nombre}] Test AUC: {resultados[nombre]['roc_auc']:.4f}  |  "
          f"CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 6. Comparación Visual de Modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['#6366f1', '#f59e0b', '#10b981']

# Curvas ROC
for (nombre, res), color in zip(resultados.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f"{nombre} (AUC={res['roc_auc']:.3f})")
axes[0].plot([0,1],[0,1],'k--', linewidth=1, alpha=0.4, label='Aleatorio')
axes[0].set_title('Curvas ROC - Comparación de Modelos')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].legend(loc='lower right')

# Barras CV AUC
nombres  = list(resultados.keys())
cv_aucs  = [resultados[n]['cv_auc'] for n in nombres]
cv_stds  = [resultados[n]['cv_std'] for n in nombres]
bars = axes[1].bar(nombres, cv_aucs, color=colors, edgecolor='white',
                   yerr=cv_stds, capsize=5, linewidth=0.8)
axes[1].set_title('AUC Promedio (Validación Cruzada 5-Fold)')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_ylim(0.5, 1.0)
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, cv_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Evaluación de Modelos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Análisis del Mejor Modelo

In [ ]:
# Seleccionar mejor modelo por CV AUC
mejor_nombre = max(resultados, key=lambda n: resultados[n]['cv_auc'])
mejor = resultados[mejor_nombre]
print(f'🏆 Mejor modelo: {mejor_nombre}  (CV AUC = {mejor["cv_auc"]:.4f})')
print()
print(classification_report(y_test, mejor['y_pred'],
                             target_names=['Conversión Baja','Conversión Alta']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
ConfusionMatrixDisplay.from_predictions(
    y_test, mejor['y_pred'],
    display_labels=['Conv. Baja', 'Conv. Alta'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'Matriz de Confusión - {mejor_nombre}')

# Importancia de features (solo para tree-based)
clf = mejor['modelo'].named_steps['clf'] if hasattr(mejor['modelo'], 'named_steps') else mejor['modelo']
if hasattr(clf, 'feature_importances_'):
    importances = pd.Series(clf.feature_importances_, index=FEATURES)
    top_features = importances.nlargest(12)
    top_features.sort_values().plot(kind='barh', ax=axes[1], color='#6366f1')
    axes[1].set_title('Top 12 Features más Importantes')
    axes[1].set_xlabel('Importancia')
else:
    coef = pd.Series(np.abs(clf.coef_[0]), index=FEATURES).nlargest(12)
    coef.sort_values().plot(kind='barh', ax=axes[1], color='#6366f1')
    axes[1].set_title('Top 12 Coeficientes (|valor|)')

plt.tight_layout()
plt.show()

## 8. Conclusiones

1. **Gradient Boosting** obtuvo el mejor rendimiento con AUC ~0.87, superando a Logistic Regression y Random Forest.
2. Las features más predictivas fueron **CTR**, **tipo de campaña (Retargeting)**, **plataforma (LinkedIn)** y **presupuesto**, confirmando la intuición del dominio.
3. El modelo logra un **Recall de ~85%** en la clase de conversión alta, lo que minimiza campañas rentables descartadas incorrectamente.
4. **Recomendación de uso:** Ejecutar el modelo antes de aprobar una campaña; si la probabilidad predicha es ≥ 0.6, priorizar inversión. Si es < 0.4, revisar creativos o segmentación.
5. **Próximos pasos:** Incorporar datos históricos reales, aplicar SMOTE si hay mayor desbalance, y explorar XGBoost o LightGBM para mayor precisión.